# Notebook 03 — dbt: модели, тесты, граф зависимостей

**Что делаем:** повторяем трансформации из Notebook 02, но теперь через **dbt**.

dbt превращает SQL в управляемый pipeline:
- версионирование через git
- автоматические тесты качества данных
- граф зависимостей (`{{ ref() }}`)
- документация

**Структура проекта:** `dbt_orders/`
```
dbt_orders/
├── dbt_project.yml
├── profiles.yml           ← DuckDB (читает Parquet)
└── models/
    ├── staging/
    │   ├── stg_orders.sql     ← нужно заполнить (TODO)
    │   ├── stg_customers.sql  ← готово
    │   └── schema.yml         ← тесты
    └── marts/
        ├── fct_revenue.sql    ← нужно заполнить (TODO)
        └── schema.yml
```

## 1. Настройка окружения

In [1]:
import os

# CWD при старте ноутбука = папка notebooks/
# LAB_DIR вычисляем ДО chdir, пока CWD ещё правильный
os.environ['LAB_DIR'] = os.path.abspath('..')   # notebooks/ → lab/

# Переходим в директорию dbt-проекта
os.chdir('../dbt_orders')

print("Рабочая директория:", os.getcwd())
print("LAB_DIR:", os.environ['LAB_DIR'])

Рабочая директория: F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab\dbt_orders
LAB_DIR: F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab


In [2]:
# Проверяем версию и подключение
!dbt --version

Core:
  - installed: 1.12.6
  - latest:    1.12.6 - Up to date!

Plugins:
  - duckdb: 1.11.1 - Up to date!


17:49:15  dbt v2 is available, and is the default for new installations. This is dbt Core v1.
          New project - install v2: https://docs.getdbt.com/docs/local/install-dbt?utm_source=dbt-cli
          Existing v1 project - upgrade guide: https://docs.getdbt.com/docs/dbt-versions/dbt-upgrade/upgrading-to-v2?utm_source=dbt-cli


In [3]:
# Проверяем соединение (profiles.yml → DuckDB)
!dbt debug --profiles-dir .

17:49:19  Running with dbt=1.12.6
17:49:19  dbt version: 1.12.6
17:49:19  python version: 3.12.0
17:49:19  python path: F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab\venv\Scripts\python.exe
17:49:19  os info: Windows-10-10.0.19043-SP0
17:49:19  Using profiles dir at .
17:49:19  Using profiles.yml file at .\profiles.yml
17:49:19  Using dbt_project.yml file at F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab\dbt_orders\dbt_project.yml
17:49:19  adapter type: duckdb
17:49:19  adapter version: 1.11.1
17:49:19  Configuration:
17:49:19    profiles.yml file [OK found and valid]
17:49:19    dbt_project.yml file [OK found and valid]
17:49:19  Required dependencies:
17:49:19   - git [OK found]

17:49:19  Connection:
17:49:19    database: warehouse
17:49:19    schema: main
17:49:19    path: F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab/data/warehouse.duckdb
17:49:19    config_options: None
17:49:19    extensions: None
17:49:19    settings: {}
17:49:19    external_root: .
17:49:19    use_credentia

## 2. Смотрим готовую модель: stg_customers

In [7]:
# т.к. я на винде
from pathlib import Path
print(Path("models/staging/stg_customers.sql").read_text(encoding="utf-8"))

-- stg_customers: чистим и типизируем сырые данные клиентов

SELECT
    customer_id,
    name                            AS customer_name,
    city,
    segment,
    created_at::date                AS signup_date
FROM read_parquet('{{ env_var("LAB_DIR", "..") }}/data/raw_customers.parquet')



## 3. Заполняем stg_orders.sql

Посмотрите на файл с TODO и заполните его.

**Требования:**
1. Переименуйте `created_at` → `loaded_at`
2. Добавьте поле `is_completed = (status = 'shipped')` (boolean)
3. Отфильтруйте строки с `amount <= 0`

In [8]:
# Смотрим текущий файл с TODO
print(Path("models/staging/stg_orders.sql").read_text(encoding="utf-8"))

-- stg_orders: стейджинг-модель заказов
--
-- TODO: Заполните SELECT-список, используя колонки из raw_orders:
--   order_id, customer_id, product, category, quantity,
--   unit_price, discount, amount, status, order_date, created_at
--
-- Требования:
--   1. Переименуйте created_at → loaded_at
--   2. Добавьте вычисляемое поле: is_completed = (status = 'shipped')
--   3. Отфильтруйте строки, где amount <= 0

SELECT
    order_id,
    customer_id,
    product,
    category,
    quantity,
    unit_price,
    discount,
    amount,
    status,
    (status = 'shipped')            AS is_completed,
    order_date,
    created_at                      AS loaded_at
FROM read_parquet('{{ env_var("LAB_DIR", "..") }}/data/raw_orders.parquet')
WHERE amount > 0



✏️ **Откройте файл** `dbt_orders/models/staging/stg_orders.sql` в редакторе и заполните SQL.

<details>
<summary>💡 Решение — раскройте после попытки</summary>

```sql
SELECT
    order_id,
    customer_id,
    product,
    category,
    quantity,
    unit_price,
    discount,
    amount,
    status,
    (status = 'shipped')            AS is_completed,
    order_date,
    created_at                      AS loaded_at
FROM read_parquet('{{ env_var("LAB_DIR", "../..") }}/data/raw_orders.parquet')
WHERE amount > 0
```
</details>

## 4. Компилируем и запускаем staging

In [9]:
# Компиляция — проверяем что SQL корректный без запуска
!dbt compile --profiles-dir . --select staging

17:54:21  Running with dbt=1.12.6
17:54:22  Registered adapter: duckdb=1.11.1
17:54:22  Unable to do partial parsing because of a version mismatch
17:54:23  [WARNING][MissingArgumentsPropertyInGenericTestDeprecation]: Deprecated
functionality
Found top-level arguments to test `accepted_values` defined on 'fct_revenue' in
package 'dbt_orders' (models\marts\schema.yml). Arguments to generic tests
should be nested under the `arguments` property.
17:54:23  Found 3 models, 12 data tests, 511 macros
17:54:23  
17:54:23  Concurrency: 2 threads (target='dev')
17:54:23  
17:54:24  [WARNING][DeprecationsSummary]: Deprecated functionality
Summary of encountered deprecations:
- MissingArgumentsPropertyInGenericTestDeprecation: 3 occurrences
To see all deprecation instances instead of just the first occurrence of each,
run command again with the `--show-all-deprecations` flag. You may also need to
run with `--no-partial-parse` as some deprecations are only encountered during
parsing.


In [10]:
# Запускаем staging модели
!dbt run --profiles-dir . --select staging

17:54:32  Running with dbt=1.12.6
17:54:33  Registered adapter: duckdb=1.11.1
17:54:33  Found 3 models, 12 data tests, 511 macros
17:54:33  
17:54:33  Concurrency: 2 threads (target='dev')
17:54:33  
17:54:33  1 of 2 START sql view model main.stg_customers ................................. [RUN]
17:54:33  2 of 2 START sql view model main.stg_orders .................................... [RUN]
17:54:33  2 of 2 OK created sql view model main.stg_orders ............................... [OK in 0.12s]
17:54:33  1 of 2 OK created sql view model main.stg_customers ............................ [OK in 0.12s]
17:54:33  
17:54:33  Finished running 2 view models in 0 hours 0 minutes and 0.32 seconds (0.32s).
17:54:33  
17:54:33  Completed successfully
17:54:33  
17:54:33  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=2


In [11]:
# Запускаем тесты качества данных
!dbt test --profiles-dir . --select staging

17:54:39  Running with dbt=1.12.6
17:54:39  Registered adapter: duckdb=1.11.1
17:54:40  Found 3 models, 12 data tests, 511 macros
17:54:40  
17:54:40  Concurrency: 2 threads (target='dev')
17:54:40  
17:54:40  1 of 8 START test accepted_values_stg_customers_segment__B2B__B2C .............. [RUN]
17:54:40  2 of 8 START test accepted_values_stg_orders_status__pending__shipped__cancelled__returned  [RUN]
17:54:40  2 of 8 PASS accepted_values_stg_orders_status__pending__shipped__cancelled__returned  [PASS in 0.10s]
17:54:40  1 of 8 PASS accepted_values_stg_customers_segment__B2B__B2C .................... [PASS in 0.10s]
17:54:40  3 of 8 START test not_null_stg_customers_customer_id ........................... [RUN]
17:54:40  4 of 8 START test not_null_stg_orders_amount ................................... [RUN]
17:54:40  3 of 8 PASS not_null_stg_customers_customer_id ................................. [PASS in 0.04s]
17:54:40  4 of 8 PASS not_null_stg_orders_amount ..........................

## 5. Смотрим что создалось в DuckDB

In [12]:
import duckdb
import os

db_path = os.path.join(os.environ['LAB_DIR'], 'data', 'warehouse.duckdb')

# Используем контекст-менеджер — соединение закрывается сразу после запроса,
# чтобы dbt мог открыть файл на запись в следующих ячейках.
with duckdb.connect(db_path) as c:
    display(c.execute("SHOW ALL TABLES").df())

,database,schema,name,column_names,column_types,temporary
0,warehouse,main,fct_revenue,"[month, segment, total_orders, total_revenue, ...","[DATE, VARCHAR, BIGINT, DOUBLE, DOUBLE, FLOAT]",False
1,warehouse,main,stg_customers,"[customer_id, customer_name, city, segment, si...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, DATE]",False
2,warehouse,main,stg_orders,"[order_id, customer_id, product, category, qua...","[BIGINT, BIGINT, VARCHAR, VARCHAR, BIGINT, DOU...",False


In [13]:
# Проверяем stg_orders
with duckdb.connect(db_path) as c:
    display(c.execute("SELECT * FROM main.stg_orders LIMIT 5").df())

,order_id,customer_id,product,category,quantity,unit_price,discount,amount,status,is_completed,order_date,loaded_at
0,1,90,Пылесос,Home,3,9000.0,0.00,27000.0,pending,False,2023-05-17,2023-05-17 11:50:00
1,2,106,Смартфон,Electronics,1,35000.0,0.00,35000.0,shipped,True,2023-08-24,2023-08-24 17:19:00
2,3,111,Куртка,Clothing,1,6500.0,0.00,6500.0,pending,False,2023-07-24,2023-07-24 16:17:00
3,4,15,Пылесос,Home,3,9000.0,0.15,22950.0,cancelled,False,2023-07-24,2023-07-24 16:01:00
4,5,56,Кофемашина,Home,3,12000.0,0.05,34200.0,shipped,True,2023-01-20,2023-01-20 09:38:00


## 6. Заполняем fct_revenue.sql

Витрина выручки — результат JOIN и агрегации staging-моделей.

**Используйте `{{ ref() }}`** вместо прямых имён таблиц — dbt строит граф зависимостей автоматически.

In [15]:
print(Path("models/marts/fct_revenue.sql").read_text(encoding="utf-8"))

-- fct_revenue: витрина выручки по месяцам и сегментам
--
-- TODO: Напишите запрос, который:
--   1. Соединяет {{ ref('stg_orders') }} с {{ ref('stg_customers') }}
--   2. Группирует по: месяцу заказа (date_trunc('month', order_date))
--      и сегменту клиента (segment)
--   3. Считает:
--      - total_orders    — количество заказов
--      - total_revenue   — сумма amount
--      - avg_order_value — средняя сумма заказа
--      - cancelled_pct   — доля cancelled (0.0 – 1.0)
--   4. Учитывает только заказы со статусом 'shipped' для total_revenue
--      (используйте FILTER или CASE WHEN)

SELECT
    date_trunc('month', o.order_date)::date          AS month,
    c.segment,
    count(*)                                         AS total_orders,
    round(sum(
        CASE WHEN o.status = 'shipped' THEN o.amount ELSE 0 END
    ), 2)                                            AS total_revenue,
    round(avg(o.amount), 2)                          AS avg_order_value,
    round(
        count(

✏️ **Откройте** `dbt_orders/models/marts/fct_revenue.sql` и напишите запрос.

<details>
<summary>💡 Решение</summary>

```sql
SELECT
    date_trunc('month', o.order_date)::date  AS month,
    c.segment,
    count(*)                                 AS total_orders,
    round(sum(
        CASE WHEN o.status = 'shipped' THEN o.amount ELSE 0 END
    ), 2)                                    AS total_revenue,
    round(avg(o.amount), 2)                  AS avg_order_value,
    round(
        count(*) FILTER (WHERE o.status = 'cancelled')::float / count(*),
        3
    )                                        AS cancelled_pct
FROM {{ ref('stg_orders') }}    AS o
JOIN {{ ref('stg_customers') }} AS c USING (customer_id)
GROUP BY 1, 2
ORDER BY 1, 2
```
</details>

In [16]:
# Запускаем всё: staging + marts
!dbt run --profiles-dir .

17:55:41  Running with dbt=1.12.6
17:55:41  Registered adapter: duckdb=1.11.1
17:55:42  Found 3 models, 12 data tests, 511 macros
17:55:42  
17:55:42  Concurrency: 2 threads (target='dev')
17:55:42  
17:55:42  1 of 3 START sql view model main.stg_customers ................................. [RUN]
17:55:42  2 of 3 START sql view model main.stg_orders .................................... [RUN]
17:55:42  2 of 3 OK created sql view model main.stg_orders ............................... [OK in 0.12s]
17:55:42  1 of 3 OK created sql view model main.stg_customers ............................ [OK in 0.12s]
17:55:42  3 of 3 START sql table model main.fct_revenue .................................. [RUN]
17:55:42  3 of 3 OK created sql table model main.fct_revenue ............................. [OK in 0.11s]
17:55:42  
17:55:42  Finished running 1 table model, 2 view models in 0 hours 0 minutes and 0.42 seconds (0.42s).
17:55:42  
17:55:42  Completed successfully
17:55:42  
17:55:42  Done. PASS=3 WA

In [17]:
# Тесты на витрине
!dbt test --profiles-dir . --select marts

17:55:47  Running with dbt=1.12.6
17:55:47  Registered adapter: duckdb=1.11.1
17:55:48  Found 3 models, 12 data tests, 511 macros
17:55:48  
17:55:48  Concurrency: 2 threads (target='dev')
17:55:48  
17:55:48  1 of 4 START test accepted_values_fct_revenue_segment__B2B__B2C ................ [RUN]
17:55:48  2 of 4 START test not_null_fct_revenue_month ................................... [RUN]
17:55:48  2 of 4 PASS not_null_fct_revenue_month ......................................... [PASS in 0.09s]
17:55:48  1 of 4 PASS accepted_values_fct_revenue_segment__B2B__B2C ...................... [PASS in 0.09s]
17:55:48  3 of 4 START test not_null_fct_revenue_segment ................................. [RUN]
17:55:48  4 of 4 START test not_null_fct_revenue_total_revenue ........................... [RUN]
17:55:48  3 of 4 PASS not_null_fct_revenue_segment ....................................... [PASS in 0.03s]
17:55:48  4 of 4 PASS not_null_fct_revenue_total_revenue ................................. 

In [18]:
# Смотрим витрину выручки
with duckdb.connect(db_path) as c:
    display(c.execute("SELECT * FROM main.fct_revenue ORDER BY month LIMIT 10").df())

,month,segment,total_orders,total_revenue,avg_order_value,cancelled_pct
0,2023-01-01,B2B,14,120045.0,25942.50,0.071
1,2023-01-01,B2C,98,1036875.0,21306.38,0.112
2,2023-02-01,B2B,21,151815.0,17910.60,0.048
3,2023-02-01,B2C,104,1752967.5,25760.22,0.125
4,2023-03-01,B2B,15,98570.0,12557.83,0.067
5,2023-03-01,B2C,122,1308412.5,17381.17,0.049
6,2023-04-01,B2B,13,432740.0,47022.31,0.000
7,2023-04-01,B2C,117,1783240.0,22333.82,0.103
8,2023-05-01,B2B,14,122865.0,12346.25,0.071
9,2023-05-01,B2C,117,1181115.0,19698.82,0.111


## 7. Граф зависимостей

In [19]:
# Генерируем документацию
!dbt docs generate --profiles-dir .
print("\nЗапустите в терминале: dbt docs serve --profiles-dir .")
print("Откройте http://localhost:8080 — там интерактивный граф зависимостей")

17:56:01  Running with dbt=1.12.6
17:56:01  Registered adapter: duckdb=1.11.1
17:56:02  Found 3 models, 12 data tests, 511 macros
17:56:02  
17:56:02  Concurrency: 2 threads (target='dev')
17:56:02  
17:56:02  Building catalog
17:56:02  Catalog written to F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab\dbt_orders\target\catalog.json

Запустите в терминале: dbt docs serve --profiles-dir .
Откройте http://localhost:8080 — там интерактивный граф зависимостей


## ✏️ Бонусное задание: новая метрика

Добавьте модель `models/marts/fct_top_products.sql` которая считает:
- топ-5 продуктов по выручке за последние 3 месяца
- с колонками: `product | category | total_revenue | order_count | avg_discount`

Добавьте тест `not_null` для `total_revenue` в `schema.yml`.

In [20]:
print(Path("models/marts/fct_top_products.sql").read_text(encoding="utf-8"))

-- Топ-5 товаров по выручке за последние 3 календарных месяца.
-- За последнюю дату берём максимальную дату заказа в наборе данных.

WITH last_month AS (
    SELECT date_trunc('month', max(order_date)) AS month
    FROM {{ ref('stg_orders') }}
)
SELECT
    o.product,
    o.category,
    round(sum(o.amount), 2) AS total_revenue,
    count(*) AS order_count,
    round(avg(o.discount), 3) AS avg_discount
FROM {{ ref('stg_orders') }} AS o
CROSS JOIN last_month AS m
WHERE o.status = 'shipped'
  AND o.order_date >= m.month - INTERVAL '2 months'
  AND o.order_date < m.month + INTERVAL '1 month'
GROUP BY o.product, o.category
ORDER BY total_revenue DESC, o.product
LIMIT 5



In [22]:
# топ-5 продуктов по выручке за последние 3 месяца
!dbt run --profiles-dir . --select fct_top_products

18:05:03  Running with dbt=1.12.6
18:05:04  Registered adapter: duckdb=1.11.1
18:05:04  Found 4 models, 13 data tests, 511 macros
18:05:04  
18:05:04  Concurrency: 2 threads (target='dev')
18:05:04  
18:05:04  1 of 1 START sql table model main.fct_top_products ............................. [RUN]
18:05:04  1 of 1 OK created sql table model main.fct_top_products ........................ [OK in 0.14s]
18:05:04  
18:05:04  Finished running 1 table model in 0 hours 0 minutes and 0.33 seconds (0.33s).
18:05:04  
18:05:04  Completed successfully
18:05:04  
18:05:04  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


In [23]:
# Проверяем total_revenue на отсутствие NULL
!dbt test --profiles-dir . --select fct_top_products

18:05:28  Running with dbt=1.12.6
18:05:29  Registered adapter: duckdb=1.11.1
18:05:29  Found 4 models, 13 data tests, 511 macros
18:05:29  
18:05:29  Concurrency: 2 threads (target='dev')
18:05:29  
18:05:29  1 of 1 START test not_null_fct_top_products_total_revenue ...................... [RUN]
18:05:29  1 of 1 PASS not_null_fct_top_products_total_revenue ............................ [PASS in 0.06s]
18:05:29  
18:05:29  Finished running 1 test in 0 hours 0 minutes and 0.21 seconds (0.21s).
18:05:29  
18:05:29  Completed successfully
18:05:29  
18:05:29  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


In [24]:
with duckdb.connect(db_path) as c:
    display(c.execute("SELECT * FROM main.fct_top_products ORDER BY total_revenue DESC").df())

,product,category,total_revenue,order_count,avg_discount
0,Ноутбук,Electronics,1443750.0,13,0.062
1,Велосипед,Sports,655200.0,14,0.064
2,Смартфон,Electronics,519750.0,9,0.050
3,Кофемашина,Home,291600.0,11,0.059
4,Наушники,Electronics,224800.0,13,0.062


In [25]:
# Обновляем документацию, чтобы в графе зависимостей появилась новая модель
!dbt docs generate --profiles-dir .

18:06:04  Running with dbt=1.12.6
18:06:04  Registered adapter: duckdb=1.11.1
18:06:05  Found 4 models, 13 data tests, 511 macros
18:06:05  
18:06:05  Concurrency: 2 threads (target='dev')
18:06:05  
18:06:05  Building catalog
18:06:06  Catalog written to F:\BMSTU\!МАГИСТРАТУРА\3_semester\Сбер\ЛР1\lab\dbt_orders\target\catalog.json


---
## Итог лабораторной работы

Мы построили мини-платформу данных:

```
Postgres (raw_orders, raw_customers)
    ↓  [Notebook 01: экспорт]
Parquet files (data/)
    ↓  [Notebook 02: DuckDB аналитика]
SQL-запросы + mart_revenue.parquet (снапшот вручную)
    ↓  [Notebook 03: dbt]
warehouse.duckdb
    ├── stg_orders     (view)   ← очистка
    ├── stg_customers  (view)   ← очистка
    └── fct_revenue    (table)  ← витрина с тестами
```

**Что такое же в production (Enterprise-масштаб):**
- Postgres → Kafka+Flink (вместо ручного экспорта)
- Parquet на MinIO → Iceberg (ACID, time travel)
- DuckDB → Trino/StarRocks (distributed)
- dbt остаётся тем же — только `profiles.yml` меняется
- Оркестрация через Dagster (FreshnessPolicy вместо cron)